# SMS Spam Detection — exploratory baseline

Adapted from the supplied notebook. Dataset: https://www.kaggle.com/datasets/tmehul/spamcsv. Download `spam.csv` to `../data/raw/spam.csv` and run this notebook from `notebooks/`. For reproducible training from the project root, use `python -m src.train`.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix


In [ ]:
dataset_path = Path("../data/raw/spam.csv")
df = pd.read_csv(dataset_path)
assert {"Category", "Message"}.issubset(df.columns)
df = df[["Category", "Message"]].dropna().copy()
df["spam"] = (df["Category"] == "spam").astype(int)
print(df.shape)
print(df["Category"].value_counts())
df.head()


## Train/test split
The uploaded original notebook used 80/20 splitting without a random seed. This cleaned version fixes a seed and stratifies labels, so results may differ from the notebook’s stored 0.99 accuracy.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df["Message"], df["spam"], test_size=0.2, random_state=42, stratify=df["spam"]
)
print(X_train.shape, X_test.shape)


## Bag-of-words encoding


In [ ]:
vectorizer = CountVectorizer()
X_train_cv = vectorizer.fit_transform(X_train)
X_test_cv = vectorizer.transform(X_test)
print("Training matrix:", X_train_cv.shape)
print("Vocabulary size:", len(vectorizer.vocabulary_))


## Multinomial Naive Bayes and evaluation


In [ ]:
model = MultinomialNB()
model.fit(X_train_cv, y_train)
y_pred = model.predict(X_test_cv)
print(classification_report(y_test, y_pred, target_names=["ham", "spam"]))
print("Confusion matrix [ham, spam]:")
print(confusion_matrix(y_test, y_pred, labels=[0, 1]))


## Example predictions
Illustrative messages, not a real mailbox or a benchmark.


In [ ]:
examples = [
    "Hey, are we meeting tomorrow?",
    "Exclusive discount! Claim your free reward today",
    "Can you send me the meeting notes?",
    "Congratulations! You won a cash prize, call now",
]
for message, pred in zip(examples, model.predict(vectorizer.transform(examples))):
    print("SPAM" if pred else "HAM", "|", message)
